# TP Final — Scoring de résiliation
## Partie A — Chargement & choix des variables

In [ ]:
# ÉTAPE 1 : Charger le fichier Excel et le convertir en CSV
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')

df = pd.read_excel('../data/dataset_assurance_ML.xlsx')
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig')

df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig')
print(df.shape)

In [ ]:
# ÉTAPE 2 : La variable cible
TARGET = 'Résiliation'
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(2))

In [ ]:
# ÉTAPE 3 : Détecter une fuite de données (data leakage)
print(pd.crosstab(df['Statut Contrat'], df[TARGET]))

In [ ]:
# ÉTAPE 4 : Choisir les variables numériques et catégorielles
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)', 
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)', 
            'Montant Sinistres (€)', 'Score Risque (0-100)']

cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre']

X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape)

In [ ]:
# ÉTAPE 5 : Vérifier le lien avec la cible
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False))
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values())

## Partie B — Pipeline, entraînement & évaluation

In [ ]:
# ÉTAPE 6 : Séparer train / test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2))

In [ ]:
# ÉTAPE 7 : Le prétraitement en un seul objet
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

In [ ]:
# ÉTAPE 8 : Deux candidats dans un Pipeline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

candidats = {
    'Régression Logistique': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=10,
        class_weight='balanced', random_state=42),
}
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
             for nom, algo in candidats.items()}

In [ ]:
# ÉTAPE 9 : Comparer par validation croisée
from sklearn.model_selection import cross_val_score

for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
# ÉTAPE 10 : Entraîner le modèle retenu et évaluer sur le test
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)

y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Accuracy :', round(float(accuracy_score(y_test, y_pred)), 3))
print('F1       :', round(float(f1_score(y_test, y_pred)), 3))
print('ROC-AUC  :', round(float(roc_auc_score(y_test, y_proba)), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

### ÉTAPE 11 : Lire la matrice de confusion comme un métier

**Q1. Un modèle qui prédirait toujours « reste » aurait 90 % d'accuracy. Est-il meilleur que le vôtre ?**
* Non. Bien qu'il ait une accuracy de 90%, il serait incapable d'identifier la moindre résiliation (0% de rappel / recall). Ce modèle serait totalement inutile pour le service Fidélisation car il ne détecterait aucun client à risque.

**Q2. Pour le service Fidélisation, quelle erreur coûte le plus cher : rater un client qui va partir (faux négatif) ou appeler un client qui serait resté (faux positif) ?**
* Rater un client qui va partir (faux négatif) coûte beaucoup plus cher, car on perd définitivement le client et la valeur de son contrat. Appeler un client qui serait resté (faux positif) coûte seulement le temps de l'appel téléphonique.

**Q3. Faut-il donc plutôt baisser ou monter le seuil de 0,5 ?**
* Il faut **baisser** le seuil (par exemple à 0.40 ou 0.35) afin d'être plus sensible et de détecter plus de clients à risque, quitte à accepter un peu plus de fausses alertes.

## Partie C — Sauvegarde & interrogation du modèle

In [ ]:
# ÉTAPE 12 : Sauvegarder le pipeline complet
import joblib, os
joblib.dump(pipeline, '../models/pipeline_resiliation.pkl')
print(os.path.getsize('../models/pipeline_resiliation.pkl') / 1024, 'Ko')

In [ ]:
# ÉTAPE 13 : Sauvegarder les métadonnées pour l'interface
import json
meta = {
    'modele': 'Random Forest',
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),
                       'median': float(X[c].median())} for c in num_cols},
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
}
with open('../models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [ ]:
# ÉTAPE 14 : Interroger le modèle sur un nouveau client
modele = joblib.load('../models/pipeline_resiliation.pkl')

print("--- PROFIL À RISQUE ---")
client_risque = pd.DataFrame([{
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3,
    'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72,
    'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur',
    'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol',
}])
print('Classe :', modele.predict(client_risque))
print('Proba  :', modele.predict_proba(client_risque)[0, 1].round(3))

print("\n--- PROFIL FIDÈLE ---")
client_fidele = pd.DataFrame([{
    'Âge': 55, 'Salaire Annuel (€)': 45000, 'Prime Annuelle (€)': 600,
    'Ancienneté (mois)': 200, 'Coeff. Bonus-Malus': 0.50, 'Nb Sinistres (3 ans)': 0,
    'Montant Sinistres (€)': 0, 'Score Risque (0-100)': 5,
    'Type Contrat': 'Gold', 'Catégorie Prof.': 'Cadre',
    'Usage Véhicule': 'Personnel', 'Dernier Sinistre': 'Aucun',
}])
print('Classe :', modele.predict(client_fidele))
print('Proba  :', modele.predict_proba(client_fidele)[0, 1].round(3))

In [ ]:
# ÉTAPE 15 : Provoquer l'erreur classique
try:
    modele.predict(client_risque.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)